# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ishaayer/flyrank-ml-internship/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [7]:
%pip -q install duckdb huggingface_hub


In [ ]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
import os
from getpass import getpass

HF_TOKEN = os.getenv("HF_TOKEN") or getpass(
    "Enter your Hugging Face token: "
)

## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [11]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [12]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [19]:

features_90d = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d
        FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,

            -- Impressions in the most recent 30 days
            SUM(CASE
                WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                THEN f.gsc_impressions ELSE 0
            END) AS imp_last30,

            -- Impressions from 30 to 60 days ago
            SUM(CASE
                WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                 AND f.report_date > b.end_d - INTERVAL 60 DAY
                THEN f.gsc_impressions ELSE 0
            END) AS imp_prev30,

            -- Impressions across the previous 60 days,
            -- excluding the most recent 30 days
            SUM(CASE
                WHEN f.report_date > b.end_d - INTERVAL 90 DAY
                 AND f.report_date <= b.end_d - INTERVAL 30 DAY
                THEN f.gsc_impressions ELSE 0
            END) AS imp_prev60,

            -- NEW: Historical search-position volatility
            STDDEV_SAMP(CASE
                WHEN f.report_date > b.end_d - INTERVAL 90 DAY
                 AND f.report_date <= b.end_d - INTERVAL 30 DAY
                THEN f.gsc_avg_position
            END) AS pos_volatility_prev60,

            -- Clicks in the most recent 30 days
            SUM(CASE
                WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                THEN f.gsc_clicks ELSE 0
            END) AS clk_last30,

            -- Average search position in the most recent 30 days
            AVG(CASE
                WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                THEN f.gsc_avg_position
            END) AS pos_last30

        FROM {TABLES['fact_daily']} f, bounds b

        WHERE f.report_date > b.end_d - INTERVAL 90 DAY

        GROUP BY 1, 2

        HAVING SUM(CASE
            WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
             AND f.report_date > b.end_d - INTERVAL 60 DAY
            THEN f.gsc_impressions ELSE 0
        END) >= 100
    )
    SELECT * FROM windowed
""").df()

print(f"90-day feature table: {len(features_90d):,} content items")
print("Columns:", features_90d.columns.tolist())

features_90d.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

90-day feature table: 111,247 content items
Columns: ['client_hash_id', 'content_hash_id', 'imp_last30', 'imp_prev30', 'imp_prev60', 'pos_volatility_prev60', 'clk_last30', 'pos_last30']


,client_hash_id,content_hash_id,imp_last30,imp_prev30,imp_prev60,pos_volatility_prev60,clk_last30,pos_last30
0,client_e547b89c05043229,content_2e296120acb03e93,2346.0,2954.0,5207.0,9.278961,0.0,38.156281
1,client_e547b89c05043229,content_516b7c0e8eec0cef,371.0,809.0,2494.0,13.766368,0.0,48.364954
2,client_e547b89c05043229,content_38b6c1a9aa29f801,5746.0,5670.0,12657.0,10.665857,2.0,40.082120
3,client_e547b89c05043229,content_2ffd36f2a70be7e3,1051.0,1567.0,3738.0,7.661814,0.0,27.835101
4,client_e547b89c05043229,content_4724385fe790d24a,4771.0,1512.0,2878.0,3.757171,11.0,8.842269


In [25]:

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# 1. Join 90-day features with query signals
data_90d = features_90d.merge(
    qsignals,
    on="content_hash_id",
    how="left",
    validate="many_to_one"
)

print("Joined rows:", f"{len(data_90d):,}")
print("Matched query signals:",
      f"{data_90d['visible_queries'].notna().sum():,}")

# 2. Create the target label
# 1 = impressions declined by more than 20%
# 0 = impressions did not decline by more than 20%
data_90d["is_declining"] = (
    data_90d["imp_last30"] < 0.8 * data_90d["imp_prev30"]
).astype(int)

# 3. Select historical predictors and query-signal features
feature_cols = [
    "imp_prev30",
    "imp_prev60",
    "pos_volatility_prev60",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share",
]

# 4. Clean missing and infinite values
data_90d[feature_cols] = data_90d[feature_cols].replace(
    [np.inf, -np.inf], np.nan
)

model_data_90d = data_90d.dropna(
    subset=feature_cols + ["is_declining", "client_hash_id"]
).copy()

X = model_data_90d[feature_cols]
y = model_data_90d["is_declining"]
groups = model_data_90d["client_hash_id"]

print("\nUsable rows:", f"{len(model_data_90d):,}")
print("Unique clients:", groups.nunique())
print("Declining share:", round(y.mean(), 3))

# 5. Split by client to test on unseen clients
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("\nTraining rows:", f"{len(X_train):,}")
print("Testing rows:", f"{len(X_test):,}")
print("Training clients:", groups.iloc[train_idx].nunique())
print("Testing clients:", groups.iloc[test_idx].nunique())

# 6. Train the model
model_90d = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model_90d.fit(X_train, y_train)

# 7. Evaluate against the majority-class baseline
predictions = model_90d.predict(X_test)

baseline_accuracy = max(
    y_test.mean(),
    1 - y_test.mean()
)

print("\nMajority-class baseline accuracy:",
      round(baseline_accuracy, 3))

print("\nUpdated model results:")
print(
    classification_report(
        y_test,
        predictions,
        digits=3,
        zero_division=0
    )
)

# 8. Inspect feature importance
importance = pd.Series(
    model_90d.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print("\nFeature importance:")
print(importance.round(4))


Joined rows: 111,247
Matched query signals: 102,203

Usable rows: 102,198
Unique clients: 49
Declining share: 0.633

Training rows: 50,498
Testing rows: 51,700
Training clients: 36
Testing clients: 13

Majority-class baseline accuracy: 0.677

Updated model results:
              precision    recall  f1-score   support

           0      0.447     0.558     0.496     16704
           1      0.761     0.670     0.713     34996

    accuracy                          0.634     51700
   macro avg      0.604     0.614     0.605     51700
weighted avg      0.659     0.634     0.643     51700


Feature importance:
imp_prev60               0.1673
pos_volatility_prev60    0.1583
anon_share               0.1582
rare_share               0.1546
imp_prev30               0.1364
top_query_share          0.1234
visible_queries          0.1017
dtype: float64


In [24]:

print("qsignals type:", type(qsignals))
print("\nColumn names:")
print(qsignals.columns.tolist())

print("\nIndex names:")
print(qsignals.index.names)

print("\nFirst 5 rows:")
display(qsignals.head())

qsignals type: <class 'pandas.core.frame.DataFrame'>

Column names:
['content_hash_id', 'visible_queries', 'rare_share', 'anon_share', 'top_query_impressions', 'kept_impressions', 'top_query_share']

Index names:
[None]

First 5 rows:


,content_hash_id,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,content_447894f2faf0d2bc,14,0.043656,0.725102,55,339.0,0.162242
1,content_4486e5efcc7b773f,10,0.041108,0.748594,169,486.0,0.347737
2,content_448b01e1de5750b4,3,0.137037,0.685185,23,48.0,0.479167
3,content_44a3108ea1a95d57,5,0.075630,0.043697,235,524.0,0.448473
4,content_44c082bdb9a864ea,23,0.004720,0.810897,542,3242.0,0.167181


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [14]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 111,247 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_e547b89c05043229,content_6b80dfab2e0ffa2e,1110.0,955.0,12.0,7.543789,1.0,0.022750,0.957216,59.0,59.0,1.000000
1,client_e547b89c05043229,content_d7bb60ec9a42c11a,3735.0,3338.0,33.0,5.446636,14.0,0.017946,0.932994,84.0,462.0,0.181818
2,client_e547b89c05043229,content_401dcc5cd616e3dd,181.0,130.0,0.0,6.874167,3.0,0.162037,0.552469,153.0,185.0,0.827027
3,client_e547b89c05043229,content_18d95bd7890430ed,151.0,340.0,0.0,33.665367,2.0,0.108932,0.820261,52.0,65.0,0.800000
4,client_e547b89c05043229,content_56f46c55f0348ab4,392.0,531.0,3.0,12.995100,5.0,0.163052,0.788332,14.0,65.0,0.215385


In [27]:

# Inspect the query table's columns
con.sql(f"""
    DESCRIBE SELECT *
    FROM {TABLES['fact_query_90d']}
""").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,query_hash_id,VARCHAR,YES,None,None,None
3,query_char_count,BIGINT,YES,None,None,None
4,query_token_count,BIGINT,YES,None,None,None
5,window_start,DATE,YES,None,None,None
6,window_end,DATE,YES,None,None,None
7,impressions_90d,BIGINT,YES,None,None,None
8,clicks_90d,BIGINT,YES,None,None,None
9,impressions_last30,BIGINT,YES,None,None,None


In [26]:

# Inspect how query signals were created
print("Query-signal columns:")
print(qsignals.columns.tolist())

print("\nQuery-signal shape:")
print(qsignals.shape)

print("\nQuery-signal sample:")
display(qsignals.head())

# Check for duplicate content IDs
print(
    "\nDuplicate content IDs:",
    qsignals["content_hash_id"].duplicated().sum()
)

# Check how many content items have query signals
print(
    "\nMatched content items:",
    features_90d["content_hash_id"].isin(
        qsignals["content_hash_id"]
    ).sum()
)


Query-signal columns:
['content_hash_id', 'visible_queries', 'rare_share', 'anon_share', 'top_query_impressions', 'kept_impressions', 'top_query_share']

Query-signal shape:
(133852, 7)

Query-signal sample:


,content_hash_id,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,content_447894f2faf0d2bc,14,0.043656,0.725102,55,339.0,0.162242
1,content_4486e5efcc7b773f,10,0.041108,0.748594,169,486.0,0.347737
2,content_448b01e1de5750b4,3,0.137037,0.685185,23,48.0,0.479167
3,content_44a3108ea1a95d57,5,0.075630,0.043697,235,524.0,0.448473
4,content_44c082bdb9a864ea,23,0.004720,0.810897,542,3242.0,0.167181



Duplicate content IDs: 0

Matched content items: 102203


In [28]:

con.sql(f"""
    SELECT
        MIN(window_start) AS earliest_window_start,
        MAX(window_start) AS latest_window_start,
        MIN(window_end) AS earliest_window_end,
        MAX(window_end) AS latest_window_end,
        COUNT(*) AS total_rows
    FROM {TABLES['fact_query_90d']}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,earliest_window_start,latest_window_start,earliest_window_end,latest_window_end,total_rows
0,2026-04-02,2026-04-02,2026-06-30,2026-06-30,2414248


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [15]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


base rate (always predict majority): 0.633
              precision    recall  f1-score   support

           0      0.547     0.341     0.420      9389
           1      0.686     0.836     0.754     16162

    accuracy                          0.654     25551
   macro avg      0.616     0.588     0.587     25551
weighted avg      0.635     0.654     0.631     25551



In [16]:

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Keep the same features and labels as the original experiment
groups = model_data["client_hash_id"]

# Split by client, not by individual content item
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_tr_group = X.iloc[train_idx]
X_te_group = X.iloc[test_idx]
y_tr_group = y.iloc[train_idx]
y_te_group = y.iloc[test_idx]

# Train on one set of clients and test on different clients
group_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

group_model.fit(X_tr_group, y_tr_group)
group_pred = group_model.predict(X_te_group)

print("Training rows:", len(X_tr_group))
print("Testing rows:", len(X_te_group))
print("Training clients:", groups.iloc[train_idx].nunique())
print("Testing clients:", groups.iloc[test_idx].nunique())
print("Baseline accuracy:",
      max(y_te_group.mean(), 1 - y_te_group.mean()))

print(classification_report(
    y_te_group, group_pred, digits=3, zero_division=0
))


Training rows: 50501
Testing rows: 51702
Training clients: 36
Testing clients: 13
Baseline accuracy: 0.6768790375614097
              precision    recall  f1-score   support

           0      0.410     0.461     0.434     16706
           1      0.726     0.683     0.704     34996

    accuracy                          0.611     51702
   macro avg      0.568     0.572     0.569     51702
weighted avg      0.624     0.611     0.617     51702



In [17]:

features_90d = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d
        FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,

            SUM(CASE
                WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                THEN f.gsc_impressions ELSE 0
            END) AS imp_last30,

            SUM(CASE
                WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                 AND f.report_date > b.end_d - INTERVAL 60 DAY
                THEN f.gsc_impressions ELSE 0
            END) AS imp_prev30,

            SUM(CASE
                WHEN f.report_date > b.end_d - INTERVAL 90 DAY
                 AND f.report_date <= b.end_d - INTERVAL 30 DAY
                THEN f.gsc_impressions ELSE 0
            END) AS imp_prev60,

            SUM(CASE
                WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                THEN f.gsc_clicks ELSE 0
            END) AS clk_last30,

            AVG(CASE
                WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                THEN f.gsc_avg_position
            END) AS pos_last30

        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 90 DAY
        GROUP BY 1, 2
        HAVING SUM(CASE
            WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
             AND f.report_date > b.end_d - INTERVAL 60 DAY
            THEN f.gsc_impressions ELSE 0
        END) >= 100
    )
    SELECT * FROM windowed
""").df()

print(f"90-day feature table: {len(features_90d):,} content items")
features_90d.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

90-day feature table: 111,247 content items


,client_hash_id,content_hash_id,imp_last30,imp_prev30,imp_prev60,clk_last30,pos_last30
0,client_e547b89c05043229,content_25dfa3e39bc37247,216.0,616.0,1060.0,1.0,12.267505
1,client_e547b89c05043229,content_bab118937886d46a,110.0,173.0,359.0,0.0,20.211330
2,client_e547b89c05043229,content_7206e9a3ceefb37a,279.0,112.0,226.0,1.0,22.038723
3,client_e547b89c05043229,content_5d6131702b65bee6,63.0,122.0,294.0,1.0,13.623843
4,client_e547b89c05043229,content_a64be0ba11772089,248.0,409.0,857.0,0.0,18.283926


Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.


Week 3+ — Results and Findings

Updated model accuracy: 63.4%

Majority-class baseline: 67.7%

Class 1 F1-score: 71.3%

New feature added: pos_volatility_prev60

Evaluation method: GroupShuffleSplit by client

Finding: The updated model achieved 63.4% accuracy, below the 67.7% majority-class baseline. Therefore, it did not outperform the baseline. The results are reported honestly without changing the metrics.